In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

In [ ]:
def set_env(wheels_path: str):
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'uninstall', 
        '--yes', 
        'keras', 
        'matplotlib', 
        'scikit-learn', 
        'tensorflow'
    ], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        wheels_path, 
        '--no-compile', 
        'vllm'
    ], check=True)

In [ ]:
set_env(wheels_path='/kaggle/input/aimo-3-utils-mk-ii/wheels')

In [ ]:
os.environ['PYTHONHASHSEED'] = '0'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VLLM_DO_NOT_TRACK'] = '1'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['VLLM_FLASH_ATTN_VERSION'] = '3'
os.environ['VLLM_LOG_STATS_INTERVAL'] = '60'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['VLLM_ATTENTION_BACKEND'] = 'FLASH_ATTN'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/input/aimo-3-utils-mk-ii/tiktoken_encodings'

In [ ]:
import gc
import re
import math
import time
import queue
import signal
import socket
import hashlib
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [ ]:
class CFG:
    
    system_prompt = (
        'You are an International Mathematical Olympiad (IMO) Gold Medalist. '
        'The answer is an integer between 0 and 99999. '
        'Place your final answer inside \\boxed{}.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code. '
        'The environment is a stateful Jupyter notebook. '
        'Use `print()` to output results.'
    )
    
    user_prompt = (
        'Use `math` and `sympy` to solve the problem, '
        'supported by `collections`, `itertools`, `fractions` and `decimal`.'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'
    
    high_problem_timeout = 996
    base_problem_timeout = 276
    
    notebook_limit = 17400
    server_timeout = 180
    
    session_timeout = 1020
    jupyter_timeout = 6
    sandbox_timeout = 3
    
    backoff_delay = 0.5
    iopub_timeout = 1.0
    
    kernel_attempts = 3
    port_increment = 10
    port_attempts = 10
    port_timeout = 300
    max_port = 65535
    min_port = 50000
    
    stream_interval = 200
    context_tokens = 81920
    batched_tokens = 2048
    entropy_floor = 0.1
    buffer_tokens = 1024
    problem_count = 50
    output_chars = 2048
    capture_size = 64
    top_logprobs = 5
    batch_size = 64
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 192
    seed = 42
    
    gpu_memory_utilization = 0.99
    temperature = 1.0
    min_p = 0.02

In [ ]:
set_seed(CFG.seed)

In [ ]:
class AIMO3Template:
    
    def __init__(self):
        
        pass
        
    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )
        
    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        
        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)
        
        user_message = Message.from_role_and_content(Role.USER, user_prompt)
        
        return [system_message, user_message]

In [ ]:
class AIMO3Sandbox:
    
    _port_lock = threading.Lock()
    _blacklisted_ports = set()
    _last_blacklist_clear = time.time()
    
    def __init__(
        self, 
        jupyter_timeout: float, 
        output_chars: int, 
        kernel_attempts: int, 
        port_increment: int, 
        port_attempts: int, 
        port_timeout: int, 
        max_port: int, 
        min_port: int, 
        backoff_delay: float, 
        iopub_timeout: float
    ):
        
        type(self)._blacklist_timeout = port_timeout
        
        if not hasattr(type(self), '_next_port'):
            type(self)._next_port = min_port
            
        if type(self)._next_port < min_port:
            type(self)._next_port = min_port
            
        self._default_timeout = jupyter_timeout
        self._output_chars = output_chars
        self._min_port = min_port
        self._max_port = max_port
        self._port_increment = port_increment
        self._port_max_attempts = port_attempts
        self._kernel_max_retries = kernel_attempts
        self._backoff_delay = backoff_delay
        self._iopub_timeout = iopub_timeout
        
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'
        
        last_exception = None
        current_delay = self._backoff_delay
        
        for retry_attempt in range(self._kernel_max_retries):
            ports = None
            
            try:
                ports = self._get_next_ports(5)
                
                self._km = KernelManager()
                self._km.shell_port = ports[0]
                self._km.iopub_port = ports[1]
                self._km.stdin_port = ports[2]
                self._km.hb_port = ports[3]
                self._km.control_port = ports[4]
                
                self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
                
                self._client = self._km.blocking_client()
                self._client.start_channels()
                self._client.wait_for_ready(timeout=self._default_timeout)
                self._owns_kernel = True
                
                break
                
            except Exception as exc:
                last_exception = exc
                
                if self._km is not None:
                    with contextlib.suppress(Exception):
                        self._km.shutdown_kernel(now=True)
                        
                    self._km = None
                    
                if ports and 'Address already in use' in str(exc):
                    self._blacklist_ports(ports)
                    
                if retry_attempt == self._kernel_max_retries - 1:
                    raise RuntimeError(f'Failed to start kernel after {self._kernel_max_retries} attempts: {exc}') from exc
                    
                time.sleep(current_delay)
                current_delay *= 2
                
        self.execute(
            'import sys\n'
            'import math\n'
            'import sympy\n'
            'import mpmath\n'
            'import decimal\n'
            'import fractions\n'
            'import itertools\n'
            'import collections\n'
            'from fractions import Fraction\n'
            'from decimal import Decimal, getcontext\n'
            'mpmath.mp.dps = 64\n'
            'getcontext().prec = 64\n'
            'sys.setrecursionlimit(4096)\n'
            'sys.set_int_max_str_digits(64)\n'
        )
        
    @classmethod
    def _is_port_available(cls, port: int) -> bool:
        
        if port in cls._blacklisted_ports:
            return False
            
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
                sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                sock.bind(('127.0.0.1', port))
                return True
                
        except OSError:
            return False
            
    @classmethod
    def _clear_old_blacklist(cls):
        
        current_time = time.time()
        
        if current_time - cls._last_blacklist_clear > cls._blacklist_timeout:
            cls._blacklisted_ports.clear()
            cls._last_blacklist_clear = current_time
            
    def _get_next_ports(self, count: int) -> list[int]:
        
        with type(self)._port_lock:
            type(self)._clear_old_blacklist()
            
            for attempt in range(self._port_max_attempts):
                start_port = type(self)._next_port
                
                candidate_ports = []
                
                for i in range(count):
                    port = start_port + i
                    
                    if port > self._max_port:
                        start_port = self._min_port
                        port = start_port + i
                        candidate_ports = [port]
                        continue
                        
                    if type(self)._is_port_available(port):
                        candidate_ports.append(port)
                        
                    else:
                        type(self)._next_port = port + 1
                        candidate_ports = []
                        break
                        
                if len(candidate_ports) == count:
                    type(self)._next_port = start_port + count
                    
                    if type(self)._next_port > self._max_port:
                        type(self)._next_port = self._min_port
                        
                    return candidate_ports
                    
                type(self)._next_port += self._port_increment
                
                if type(self)._next_port > self._max_port:
                    type(self)._next_port = self._min_port
                    
            raise RuntimeError(f'Unable to find {count} available ports after {self._port_max_attempts} attempts')
            
    def _blacklist_ports(self, ports: list[int]):
        
        with type(self)._port_lock:
            type(self)._blacklisted_ports.update(ports)
            
    def _format_error(self, traceback: list[str]) -> str:
        
        clean_lines = []
        
        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            
            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue
                
            clean_lines.append(clean_frame)
            
        return ''.join(clean_lines)
        
    def _truncate_output(self, text: str) -> str:
        
        if len(text) < self._output_chars:
            return text
            
        slice_chars = self._output_chars // 2
        
        return (
            f'{text[:slice_chars]}\n'
            f'... [Truncated {len(text) - self._output_chars} characters] ...\n'
            f'{text[-slice_chars:]}'
        )
        
    def execute(self, code: str, timeout: float | None = None) -> str:
        
        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )
        
        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()
        
        while True:
            elapsed = time.time() - start_time
            
            if elapsed > effective_timeout:
                self._km.interrupt_kernel()
                
                return f'[ERROR] Execution timed out after {effective_timeout} seconds'
                
            try:
                msg = client.get_iopub_msg(timeout=self._iopub_timeout)
                
            except queue.Empty:
                continue
                
            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue
                
            msg_type = msg.get('msg_type')
            content = msg.get('content', {})
            
            if msg_type == 'stream':
                text = content.get('text', '')
                
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                    
                else:
                    stderr_parts.append(text)
                    
            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])
                
                stderr_parts.append(self._format_error(traceback_list))
                
            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')
                
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')
                    
            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break
                    
        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)
        
        stdout = self._truncate_output(stdout)
        
        if stderr:
            stderr = self._truncate_output(stderr)
            
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
            
        return stdout if stdout.strip() else '[WARNING] No output. Use `print()` to output results.'
        
    def _kill_descendants(self, pid: int) -> None:
        
        children = []
        
        for entry in os.listdir('/proc'):
            if not entry.isdigit():
                continue
                
            try:
                with open(f'/proc/{entry}/stat') as f:
                    stat = f.read()
                    ppid = int(stat[stat.rfind(')') + 2:].split()[1])
                    
                    if ppid == pid:
                        children.append(int(entry))
                        
            except (FileNotFoundError, ProcessLookupError, PermissionError, ValueError, IndexError):
                pass
                
        for child_pid in children:
            self._kill_descendants(child_pid)
            
            try:
                os.kill(child_pid, signal.SIGKILL)
                
            except (ProcessLookupError, PermissionError):
                pass
                
    def reset(self):
        
        if self._km and self._km.is_alive():
            self._km.interrupt_kernel()
            
            try:
                self._kill_descendants(self._km.kernel.pid)
                
            except Exception:
                pass
                
        self.execute(
            '%reset -f\n'
            'import sys\n'
            'import math\n'
            'import sympy\n'
            'import mpmath\n'
            'import decimal\n'
            'import fractions\n'
            'import itertools\n'
            'import collections\n'
            'from fractions import Fraction\n'
            'from decimal import Decimal, getcontext\n'
            'mpmath.mp.dps = 64\n'
            'getcontext().prec = 64\n'
            'sys.setrecursionlimit(4096)\n'
            'sys.set_int_max_str_digits(64)\n'
        )
        
    def close(self):
        
        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()
                
        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
                
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

In [ ]:
class AIMO3Tool:
    
    def __init__(self, tool_prompt: str, sandbox):
        
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        self._execution_lock = threading.Lock()
        
    @property
    def instruction(self) -> str:
        
        return self._tool_prompt
        
    @property
    def tool_config(self) -> ToolNamespaceConfig:
        
        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )
        
    def _make_response(self, output: str, channel: str | None = None) -> Message:
        
        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')
        
        if channel:
            message = message.with_channel(channel)
            
        return message
        
    def process_sync_plus(self, message: Message) -> list[Message]:
        
        script = message.content[0].text
        
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(script)
                
            except TimeoutError as exc:
                output = f'[ERROR] {exc}'
                
        return [self._make_response(output, channel=message.channel)]

In [ ]:
class AIMO3Solver:
    
    def __init__(self, cfg, port: int = 8000):
        
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
        
        self.boxed_pattern = re.compile(r'\\boxed\s*\{\s*([0-9,_]+)\s*\}')
        
        self._preload_model_weights()        
        self.server_process = self._start_server()
        
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
        
        self._wait_for_server()
        self._initialize_kernels()
        
        self.notebook_start_time = time.time()
        self.problems_remaining = self.cfg.problem_count
        
    def _preload_model_weights(self) -> None:
        
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
        
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
                
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
                    
        def _read_file(path: str) -> None:
            
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
                    
        with ThreadPoolExecutor() as executor:
            list(executor.map(_read_file, files_to_load))
            
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
        
    def _start_server(self) -> subprocess.Popen:
        
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--max-num-batched-tokens', 
            str(self.cfg.batched_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--max-cudagraph-capture-size', 
            str(self.cfg.capture_size), 
            '--max-logprobs', 
            str(self.cfg.top_logprobs), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
        
        self.log_file = open('vllm_server.log', 'w')
        
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
        
    def _wait_for_server(self):
        
        print('Waiting for vLLM server...')
        start_time = time.time()
        
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
            
            if return_code is not None:
                self.log_file.flush()
                
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
                    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
                
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                
                return
                
            except Exception:
                time.sleep(1)
                
        raise RuntimeError('Server failed to start (timeout).\n')
        
    def _initialize_kernels(self) -> None:
        
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
        
        self.sandbox_pool = queue.Queue()
        
        def _create_sandbox():
            
            return AIMO3Sandbox(
                jupyter_timeout=self.cfg.jupyter_timeout, 
                output_chars=self.cfg.output_chars, 
                kernel_attempts=self.cfg.kernel_attempts, 
                port_increment=self.cfg.port_increment, 
                port_attempts=self.cfg.port_attempts, 
                port_timeout=self.cfg.port_timeout, 
                max_port=self.cfg.max_port, 
                min_port=self.cfg.min_port, 
                backoff_delay=self.cfg.backoff_delay, 
                iopub_timeout=self.cfg.iopub_timeout
            )
            
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
            
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
                
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
        
    def _scan_for_answer(self, text: str) -> int | None:
        
        matches = self.boxed_pattern.findall(text)
        
        if matches:
            try:
                clean_value = re.sub(r'[,_]', '', matches[-1])
                value = int(clean_value)
                
                if 0 <= value <= 99999:
                    return value
                    
            except ValueError:
                pass
                
        return None
        
    def _stream_completion(
        self, 
        prompt_ids: list, 
        attempt_seed: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> tuple[list[int], list[str], float, int]:
        
        max_tokens = self.cfg.context_tokens - len(prompt_ids)
        
        if max_tokens < self.cfg.buffer_tokens:
            return [], [], 0.0, 0
            
        stream = self.client.completions.create(
            model=self.cfg.served_model_name, 
            temperature=self.cfg.temperature, 
            logprobs=self.cfg.top_logprobs, 
            max_tokens=max_tokens, 
            prompt=prompt_ids, 
            seed=attempt_seed, 
            stream=True, 
            extra_body={
                'min_p': self.cfg.min_p, 
                'stop_token_ids': self.stop_token_ids, 
                'return_token_ids': True
            }
        )
        
        token_buffer = []
        text_chunks = []
        
        cumulative_entropy = 0.0
        entropy_token_count = 0
        
        try:
            for chunk in stream:
                if stop_event.is_set() or time.time() > deadline:
                    break
                    
                new_tokens = chunk.choices[0].token_ids
                new_text = chunk.choices[0].text
                
                if new_tokens:
                    token_buffer.extend(new_tokens)
                    text_chunks.append(new_text)
                    
                    chunk_logprobs = chunk.choices[0].logprobs
                    
                    if chunk_logprobs is not None:
                        if chunk_logprobs.top_logprobs:
                            for top_logprobs_dict in chunk_logprobs.top_logprobs:
                                token_entropy = 0.0
                                
                                for log_prob in top_logprobs_dict.values():
                                    prob = math.exp(log_prob)
                                    
                                    if prob > 0:
                                        token_entropy -= prob * math.log2(prob)
                                        
                                cumulative_entropy += token_entropy
                                entropy_token_count += 1
                                
        finally:
            stream.close()
            
        return token_buffer, text_chunks, cumulative_entropy, entropy_token_count
        
    def _handle_tool_call(self, last_message: Message, local_tool: AIMO3Tool) -> tuple[int, list[Message], str]:
        
        tool_responses = local_tool.process_sync_plus(last_message)
        response_text = tool_responses[0].content[0].text
        
        python_errors = 0
        
        if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
            python_errors = 1
            
        return python_errors, tool_responses, response_text
        
    def _process_attempt(
        self, 
        user_text: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
        
        start_time = time.time()
        
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Input Tokens': 0, 
                'Python Tokens': 0, 
                'Response Length': 0, 
                'Total Tokens': 0, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Entropy': float('inf'), 
                'Time': '00:00', 
                'From Python': False, 
                'Answer': None
            }
            
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        response_tokens = 0
        final_answer = None
        
        input_tokens = 0
        python_tokens = 0
        
        total_entropy = 0.0
        total_entropy_tokens = 0
        
        python_outputs = set()
        
        string = f'{self.cfg.seed}-{attempt_index}'
        signature = hashlib.sha256(string.encode()).hexdigest()
        
        attempt_seed = int(signature, 16) % (2**31)
        
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
            
            local_tool = AIMO3Tool(
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
            
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                user_text, 
                local_tool.tool_config
            )
            
            conversation = Conversation.from_messages(messages)
            
            initial_prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
            input_tokens = len(initial_prompt_ids)
            
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
                    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                
                token_buffer, text_chunks, chunk_entropy, chunk_tokens = self._stream_completion(
                    prompt_ids, 
                    attempt_seed, 
                    stop_event, 
                    deadline
                )
                
                response_tokens += len(token_buffer)
                total_entropy += chunk_entropy
                total_entropy_tokens += chunk_tokens
                
                if not token_buffer:
                    break
                    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
                
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    
                    last_brace_index = answer_text.rfind('}')
                    
                    if last_brace_index != -1:
                        search_text = answer_text[:last_brace_index + 1]
                        final_answer = self._scan_for_answer(search_text)
                        
                    break
                    
                if last_message.recipient == 'python':
                    python_calls += 1
                    
                    errors, tool_responses, response_text = self._handle_tool_call(last_message, local_tool)
                    python_errors += errors
                    
                    for integer_match in re.finditer(r'\b([0-9,_]+)\b', response_text):
                        try:
                            value = int(integer_match.group(1))
                            
                            if 0 <= value <= 99999:
                                python_outputs.add(value)
                                
                        except ValueError:
                            pass
                            
                    for tool_msg in tool_responses:
                        if tool_msg.content and hasattr(tool_msg.content[0], 'text'):
                            text = tool_msg.content[0].text
                            python_tokens += len(encoding.encode(text))
                            
                    conversation.messages.extend(tool_responses)
                    
        except Exception as exc:
            python_errors += 1
            
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
                
        from_python = final_answer is not None and final_answer in python_outputs
        
        total_tokens = input_tokens + python_tokens + response_tokens
        
        if total_entropy_tokens > 0:
            mean_entropy = total_entropy / total_entropy_tokens
            
        else:
            mean_entropy = float('inf')
            
        elapsed = time.time() - start_time
        time_duration = f'{int(elapsed // 60):02d}:{int(elapsed % 60):02d}'
        
        return {
            'Attempt': attempt_index + 1, 
            'Input Tokens': input_tokens, 
            'Python Tokens': python_tokens, 
            'Response Length': response_tokens, 
            'Total Tokens': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Time': time_duration, 
            'From Python': from_python, 
            'Answer': final_answer
        }
        
    def _select_answer(self, detailed_results: list) -> int:
        
        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)
        
        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1 / max(entropy, self.cfg.entropy_floor)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1
                
        scored_answers = []
        
        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })
            
        scored_answers.sort(key=lambda x: x['score'], reverse=True)
        
        vote_data = []
        
        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))
            
        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )
        
        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0
            
        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')
        
        return final_answer
        
    def solve_problem(self, id_value: int, problem: str) -> int:
        
        print(f'\n({id_value}) {problem}\n')
        
        user_text = f'{problem} {self.cfg.user_prompt}'
        
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
        
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
        
        deadline = time.time() + budget
        
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
        
        tasks = []
        
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
            
        detailed_results = []
        valid_answers = []
        
        stop_event = threading.Event()
        
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
        
        try:
            futures = []
            
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_text, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
                
                futures.append(future)
                
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
                        
                    counts = Counter(valid_answers).most_common(1)
                    
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
                        break
                        
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
                    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
            
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
            
        if not valid_answers:
            print('\nResult: 0\n')
            
            return 0
            
        return self._select_answer(detailed_results)

In [ ]:
solver = AIMO3Solver(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
    id_value = id_.item(0)
    question_text = question.item(0)
    
    gc.disable()
    
    final_answer = solver.solve_problem(id_value, question_text)
    
    gc.enable()
    gc.collect()
    
    return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [ ]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv',)
    )